# Day 20: Deep Research Agent with Multi-Model Orchestration

In this notebook, we'll build a complete Deep Research Agent that:
- Uses multiple AI models (OpenAI, Gemini, DeepSeek, Grok)
- Implements structured outputs with Pydantic
- Adds guardrails for input validation
- Performs parallel web searches
- Synthesizes findings into a comprehensive report
- Sends results via email

**Note**: This notebook uses the OpenAI Agents SDK (not Swarm). Make sure you have your API keys set up in a `.env` file.


## Setup and Imports


In [4]:
# Import required libraries
import os
import asyncio
import json
from dotenv import load_dotenv
from openai import OpenAI, AsyncOpenAI
from pydantic import BaseModel
from IPython.display import Markdown, display

# Load environment variables
load_dotenv(override=True)

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✓ Environment loaded")
print(f"✓ OpenAI API Key: {'Set' if os.getenv('OPENAI_API_KEY') else 'Missing'}")
print("✓ OpenAI client initialized")


✓ Environment loaded
✓ OpenAI API Key: Set
✓ OpenAI client initialized


## Part 1: Structured Outputs with Pydantic

Define schemas that the LLM must follow when generating responses.


In [5]:
# Define structured output schemas
class WebSearchItem(BaseModel):
    reason: str
    query: str

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem]

class ReportData(BaseModel):
    short_summary: str
    markdown_report: str
    follow_up_suggestions: list[str]

print("✓ Schemas defined")


✓ Schemas defined


## Part 2: Research Planning

Convert a research query into specific search queries.


In [6]:
NUM_SEARCHES = 3  # Configurable (start with 3 to control costs)

def plan_searches(query: str) -> WebSearchPlan:
    """Generate a plan of web searches for the query."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": f"""You are a research assistant.
Given a query, come up with {NUM_SEARCHES} web searches to perform.
Respond with JSON: {{"searches": [{{"reason": "...", "query": "..."}}]}}"""
            },
            {"role": "user", "content": query}
        ],
        response_format={"type": "json_object"}
    )
    
    result = json.loads(response.choices[0].message.content)
    searches = [WebSearchItem(**item) for item in result["searches"]]
    return WebSearchPlan(searches=searches)

# Test the planner
test_query = "Latest AI agent frameworks in 2025"
plan = plan_searches(test_query)

print(f"Search plan for: {test_query}\n")
for i, search in enumerate(plan.searches, 1):
    print(f"{i}. {search.query}")
    print(f"   Reason: {search.reason}\n")


Search plan for: Latest AI agent frameworks in 2025

1. latest AI agent frameworks 2025
   Reason: To find the most recent advancements and frameworks for AI agents that are expected to be prominent in 2025.

2. new AI frameworks companies 2025
   Reason: To gather information on companies and projects releasing new AI frameworks or tools in the upcoming years.

3. future of AI agents 2025 predictions
   Reason: To explore predictions and expert opinions on the future of AI agent development and technologies by 2025.



## Part 3: Simulated Web Search

**Note**: OpenAI's Web Search Tool costs ~$0.025 per search. For this demo, we'll simulate searches to avoid costs. In production, you would use the OpenAI Agents SDK's `WebSearchTool`.


In [7]:
def perform_web_search(search_item: WebSearchItem) -> str:
    """Simulate a web search (in production, use OpenAI's WebSearchTool)."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": """You're a research assistant simulating web search results.
Produce a concise 2-3 paragraph summary of what you'd find searching for this query."""
            },
            {
                "role": "user",
                "content": f"Search: {search_item.query}\nReason: {search_item.reason}"
            }
        ]
    )
    return response.choices[0].message.content

# Test single search
result = perform_web_search(plan.searches[0])
print(f"Search result sample:\n{result[:200]}...")


Search result sample:
As of late 2023, the landscape for AI agent frameworks is rapidly evolving, with several key trends and advancements expected to shape the field by 2025. Notable frameworks include the open-source pro...


## Part 4: Parallel Search Execution

Use asyncio to run all searches in parallel for efficiency.


In [8]:
async def perform_all_searches(search_plan: WebSearchPlan) -> list[str]:
    """Execute all searches in parallel."""
    async def search_async(item: WebSearchItem) -> str:
        # In a real async implementation, this would be truly async
        return perform_web_search(item)
    
    tasks = [search_async(item) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    return results

# Execute all searches
search_results = await perform_all_searches(plan)
print(f"✓ Completed {len(search_results)} searches in parallel")


✓ Completed 3 searches in parallel


## Part 5: Report Writing

Synthesize all search results into a comprehensive report.


In [9]:
def write_report(query: str, search_results: list[str]) -> ReportData:
    """Synthesize search results into a comprehensive report."""
    combined_results = "\n\n".join([
        f"Search Result {i+1}:\n{result}"
        for i, result in enumerate(search_results)
    ])
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": """You are a senior researcher writing a cohesive report.
Create a detailed markdown report (1000+ words) with:
- Introduction and overview
- Main findings with sections
- Conclusion
Respond with JSON: {"short_summary": "...", "markdown_report": "...", "follow_up_suggestions": [...]}"""
            },
            {
                "role": "user",
                "content": f"Query: {query}\n\nResearch Results:\n{combined_results}"
            }
        ],
        response_format={"type": "json_object"}
    )
    
    result = json.loads(response.choices[0].message.content)
    return ReportData(**result)

# Generate report
report = write_report(test_query, search_results)
print(f"✓ Report generated ({len(report.markdown_report)} characters)")
print(f"\nShort Summary:\n{report.short_summary}")


✓ Report generated (5038 characters)

Short Summary:
The report explores the anticipated advancements in AI agent frameworks by 2025, emphasizing multimodal capabilities, ethical AI development, and personalization through cutting-edge technologies like reinforcement learning and generative AI. Key players include tech giants and innovative startups, indicating a diverse landscape for AI development.


In [10]:
# Display the full report
print("Full Report:")
print("=" * 80)
display(Markdown(report.markdown_report))


Full Report:


# Latest AI Agent Frameworks in 2025: An Overview

## Introduction
The evolution of AI agents represents one of the most exciting frontiers in technology today. By 2025, the capabilities of these intelligent systems are projected to expand significantly, with various frameworks setting the stage for enhanced functionality, interactivity, and ethical considerations. The focus of this report is to highlight the state-of-the-art developments in AI frameworks expected to gain traction by 2025, examining trends, innovations, and challenges that may arise.

## Main Findings

### 1. Emergence of Multimodal AI Frameworks
Recent advancements indicate a growing trend toward multimodal AI frameworks that integrate various forms of data, such as text, images, and audio.

- **Integration of Natural Language Processing (NLP) and Computer Vision**: Companies like OpenAI and Google are at the forefront, enhancing interaction models that combine NLP with computer vision and robotics. This fusion allows AI agents to interpret and interact with their surroundings more naturally, enhancing user experience.
- **Real-Time Decision Making**: Innovations are also leading to AI agents that can make decisions in real time based on input across multiple modalities, significantly improving their utility in complex task execution.

### 2. Advances in Reinforcement Learning
Reinforcement learning (RL) continues to be a pivotal area of research for developing more adaptive and capable AI agents.

- **Learning from Environment**: Future frameworks will allow agents to learn from their interactions with the environment, adjusting their behavior to improve performance iteratively. This learning paradigm is essential for tasks ranging from gaming to real-world applications like logistics and transportation.
- **Increased Personalization**: Personalization features, driven by RL methodologies, aim to create agents that can understand user preferences and emotional cues, further enhancing engagement and interaction.

### 3. Focus on Ethical AI Development
As AI technologies become more embedded in everyday life, the industry recognizes the importance of developing frameworks that are ethical and responsible.

- **Transparency and Accountability**: Emphasizing transparency in AI processes aims to maintain user trust, with frameworks designed to disclose how decisions are made and to prevent biases that could skew data interpretations.
- **Regulatory Compliance**: By 2025, the frameworks are expected to align with emerging regulations concerning data privacy and ethical use, thus paving the way for secure AI implementations.

### 4. Adoption and Innovation from Established Players and Startups
The landscape of AI frameworks is intensely competitive, featuring both tech giants and nimble startups.

- **Continuous Improvement from Tech Giants**: Companies like Google, Microsoft, and IBM are expected to enhance their offerings significantly. For example, Google will improve TensorFlow with an eye toward integrating cloud services to support scalable applications, while Microsoft is likely to bolster its Azure AI services with more accessible tools for developers.
- **Contributions from Startups**: Emerging players, particularly in the NLP domain, are making strides with user-friendly tools that democratize access to advanced AI capabilities, such as those offered by Hugging Face.

### 5. Open-Source Frameworks and Community Collaboration
The momentum behind open-source AI frameworks is growing, with community contributions providing essential innovations.

- **Collaborative Efforts**: Collaborative projects are expected to gain traction, allowing for a rapid adaptation of frameworks to meet the diverse needs of end-users while fostering innovation through shared knowledge and jointly developed technologies.
- **Flexibility and Adaptability**: Open-source status affords the ability to customize frameworks for niche applications, making them highly adaptable in a fast-changing technological environment.

## Conclusion
Looking ahead to 2025, the field of AI agent frameworks is poised for dynamic growth and transformation. The integration of multimodal capabilities, reinforcement learning, and ethical considerations will significantly shape how AI agents function and interact with users. The competitive landscape will witness contributions from established corporations alongside innovative startups, while the push for open-source frameworks will enhance innovation and accessibility.

As artificial intelligence increasingly permeates everyday life and business operations, careful attention must be given to ethical and responsible AI development. By fostering transparency and community engagement, the future of AI agents can lead to systems that are not only powerful but also aligned with human values and societal needs. The next few years will undoubtedly present a wealth of opportunities and challenges that will redefine our understanding of intelligent systems and their role in our world.

## Part 6: Email Delivery

Send the report via email using SendGrid.


In [11]:
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail

def send_email(subject: str, html_body: str) -> str:
    """Send email via SendGrid."""
    # IMPORTANT: Update these with your verified emails!
    from_email = "your-verified@email.com"
    to_emails = "recipient@email.com"
    
    message = Mail(
        from_email=from_email,
        to_emails=to_emails,
        subject=subject,
        html_content=html_body
    )
    
    try:
        sg = SendGridAPIClient(os.getenv("SENDGRID_API_KEY"))
        response = sg.send(message)
        return f"✓ Email sent! Status: {response.status_code}"
    except Exception as e:
        return f"✗ Error: {str(e)}"

def convert_to_html(report: ReportData) -> tuple[str, str]:
    """Convert markdown report to HTML with subject line."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": """Convert this markdown report to clean HTML.
Also create an appropriate subject line.
Respond with JSON: {"subject": "...", "html_body": "..."}"""
            },
            {"role": "user", "content": report.markdown_report}
        ],
        response_format={"type": "json_object"}
    )
    
    result = json.loads(response.choices[0].message.content)
    return result["subject"], result["html_body"]

# Uncomment to send email:
# subject, html_body = convert_to_html(report)
# result = send_email(subject, html_body)
# print(result)

print("✓ Email functions defined (sending commented out)")


✓ Email functions defined (sending commented out)


## Part 7: Complete Workflow

Put it all together into one function that orchestrates the entire research process.


In [12]:
async def run_deep_research(query: str, send_email_flag: bool = False):
    """Complete deep research workflow."""
    print(f"🔍 Starting deep research: {query}")
    print("=" * 80)
    
    # Step 1: Plan
    print("\n📋 Planning searches...")
    search_plan = plan_searches(query)
    print(f"   Generated {len(search_plan.searches)} queries")
    
    # Step 2: Search
    print(f"\n🌐 Performing {len(search_plan.searches)} searches...")
    search_results = await perform_all_searches(search_plan)
    print("   ✓ Searches complete")
    
    # Step 3: Write
    print("\n📝 Writing report...")
    report = write_report(query, search_results)
    print(f"   ✓ Report generated ({len(report.markdown_report)} chars)")
    
    # Step 4: Email (optional)
    if send_email_flag:
        print("\n📧 Sending email...")
        subject, html_body = convert_to_html(report)
        result = send_email(subject, html_body)
        print(f"   {result}")
    
    print("\n" + "=" * 80)
    print("✅ Research complete!\n")
    
    return report

print("✓ Complete workflow function defined")


✓ Complete workflow function defined


In [13]:
# Run the complete research workflow
research_query = "What are the most exciting commercial applications of autonomous agentic AI as of 2025?"

# Execute (send_email_flag=False to avoid sending email)
final_report = await run_deep_research(research_query, send_email_flag=False)

# Display the report
print("\n" + "=" * 80)
print("FINAL REPORT")
print("=" * 80)
display(Markdown(final_report.markdown_report))

print("\n" + "=" * 80)
print("FOLLOW-UP SUGGESTIONS")
print("=" * 80)
for i, suggestion in enumerate(final_report.follow_up_suggestions, 1):
    print(f"{i}. {suggestion}")


🔍 Starting deep research: What are the most exciting commercial applications of autonomous agentic AI as of 2025?

📋 Planning searches...
   Generated 3 queries

🌐 Performing 3 searches...
   ✓ Searches complete

📝 Writing report...
   ✓ Report generated (5728 chars)

✅ Research complete!


FINAL REPORT


# Report on Commercial Applications of Autonomous Agentic AI in 2025

## Introduction and Overview
As we approach the year 2025, the commercial landscape for autonomous agentic AI is increasingly dynamic, affecting a broad range of industries. Autonomous agents, which include AI systems capable of decision-making and independent task execution, have transformed operations, customer engagement, and overall efficiency in multiple sectors. This report delves into the most exciting commercial applications of autonomous agentic AI as of 2025, highlighting key industries where significant advancements have occurred.

The intersection of technological innovation and increasing sophistication in machine learning, natural language processing, and robotics provides a solid framework for testing and implementing AI solutions. However, this rapid evolution also requires careful consideration of ethical implications, including governance and accountability. Essentially, while autonomous AI provides remarkable opportunities, it also necessitates a thoughtful approach to its integration into society. 

## Main Findings

### 1. Logistics and Supply Chain Management
One of the most profound applications of autonomous agentic AI has emerged in logistics and supply chain management. Companies such as Amazon and Alibaba exemplify how intelligent automation can optimize routing, inventory control, and demand forecasting. By employing autonomous agents, these companies can enhance their fulfillment processes
- **Routing Optimization**: AI agents analyze vast datasets to identify the best delivery routes, resulting in significant cost savings and enhanced efficiency.
- **Inventory Management**: Autonomous AIs predict inventory requirements, effectively reducing waste and improving stock levels. 
- **Demand Forecasting**: These systems enable companies to better align their supply with market demand, ensuring that products are available when needed without overstocking. 

### 2. Financial Services
In the financial sector, autonomous agents have revolutionized traditional models of investing and banking through the emergence of AI-driven robo-advisors. These agents offer personalized investment strategies through:
- **Data Analysis**: They can analyze vast amounts of financial data to uncover trends, enabling adaptive and responsive investment strategies.
- **Risk Management**: Autonomous AIs assess risks and execute trades in real-time, democratizing access to sophisticated financial services for individuals.
- **Fraud Detection**: AI technology enhances fraud detection systems, analyzing transaction patterns to identify anomalies more effectively. 

### 3. Healthcare Innovations
The healthcare industry has reaped the benefits of autonomous agentic AI through advanced applications that improve patient care:
- **Autonomous Diagnosis**: AI systems can analyze medical data and diagnostic images, assisting doctors in identifying health issues early. 
- **Patient Monitoring**: Continuous monitoring with AI-powered systems enables proactive interventions in patient care, which contributes to better health outcomes. 
- **Personalized Treatment**: These agents provide tailored treatment recommendations rooted in thorough data analysis, enhancing patient experience and recovery rates. 

### 4. Consumer Engagement and Marketing
Autonomous AIs are also changing the way businesses engage with consumers, particularly through:
- **Chatbots and Virtual Assistants**: Significant advancements in natural language processing have allowed chatbots to handle complex customer inquiries efficiently. This capability improves user experience while reducing the workload on human agents.
- **Personalized Experiences**: AI systems analyze consumer behavior to provide tailored product recommendations, thereby enhancing marketing effectiveness and customer satisfaction.
- **Feedback and Sentiment Analysis**: Businesses can utilize AI for real-time feedback collection and sentiment analysis, allowing for agile adjustments to marketing strategies and product offerings.

### 5. Ethical and Regulatory Landscape
As the integration of autonomous AI systems becomes more ubiquitous, ethical considerations surrounding governance, accountability, and safety are coming to the forefront:
- **Regulatory Frameworks**: The development of governance structures is critical to ensuring that autonomous agents operate responsibly and ethically.
- **Employment Implications**: The rise of AI agents poses potential disruptions in employment, necessitating discussions about retraining and the evolution of the workforce.
- **Societal Norms**: Ongoing discussions regarding privacy, autonomy, and the psychological effects of interacting with AI agents are crucial for determining the future role of autonomous AIs in daily life. 

## Conclusion
In summary, the advancements in autonomous agentic AI by 2025 are making a considerable impact across various industries including logistics, finance, healthcare, and marketing. The proliferation of AI technologies that support enhanced decision-making and operational efficiency presents exciting commercial opportunities. However, it is essential to navigate the accompanying ethical landscape thoughtfully, ensuring that innovation aligns with societal values and public welfare. By fostering dialogues between stakeholders in technology, policy, and society, it is possible to harness the full potential of autonomous AI while promoting responsible and ethical practices in its implementation. As we move forward, continued research and engagement will be crucial in shaping a future where autonomous AIs enhance human experiences without compromising ethical standards.


FOLLOW-UP SUGGESTIONS
1. Research on regulatory frameworks for autonomous AI
2. Explore ethical implications of AI in job markets
3. Case studies of companies successfully integrating autonomous agents
4. Investigate public perception of autonomous AI technologies
5. Examine future technological advancements in AI capabilities


## Part 8: Enhancements

Here are some ways to improve the Deep Research Agent.


In [14]:
# Enhancement 1: Clarifying Questions
class ClarifyingQuestions(BaseModel):
    questions: list[str]

def generate_clarifying_questions(query: str) -> ClarifyingQuestions:
    """Generate clarifying questions to better understand the research needs."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": """Given a research query, generate 3 clarifying questions that would help
produce a more targeted and useful research report.
Respond with JSON containing a 'questions' array."""
            },
            {"role": "user", "content": query}
        ],
        response_format={"type": "json_object"}
    )
    
    result = json.loads(response.choices[0].message.content)
    return ClarifyingQuestions(questions=result["questions"])

# Test
questions = generate_clarifying_questions("AI agent frameworks")
print("Clarifying Questions:")
for i, q in enumerate(questions.questions, 1):
    print(f"{i}. {q}")


Clarifying Questions:
1. What specific applications or industries are you interested in regarding AI agent frameworks?
2. Are you looking for information on specific types of AI agents, such as conversational agents, autonomous agents, or robotic agents?
3. Would you like to focus on a particular programming language or platform for implementing AI agent frameworks?


In [15]:
# Enhancement 2: Report Evaluation
class EvaluationResult(BaseModel):
    quality_score: int
    strengths: list[str]
    gaps: list[str]
    improvement_suggestions: list[str]

def evaluate_report(query: str, report: ReportData) -> EvaluationResult:
    """Evaluate the quality of a research report."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": """You are a senior research evaluator.
Review the report for completeness, accuracy, clarity, and usefulness.
Identify strengths, gaps, and provide specific improvement suggestions.
Respond with JSON containing 'quality_score', 'strengths', 'gaps', and 'improvement_suggestions'."""
            },
            {
                "role": "user",
                "content": f"Query: {query}\n\nReport:\n{report.markdown_report}"
            }
        ],
        response_format={"type": "json_object"}
    )
    
    result = json.loads(response.choices[0].message.content)
    return EvaluationResult(**result)

# Test evaluation
evaluation = evaluate_report(research_query, final_report)
print(f"Quality Score: {evaluation.quality_score}/10")
print(f"\nStrengths: {len(evaluation.strengths)}")
print(f"Gaps: {len(evaluation.gaps)}")
print(f"Suggestions: {len(evaluation.improvement_suggestions)}")


Quality Score: 8/10

Strengths: 4
Gaps: 3
Suggestions: 4


## Summary

In this notebook, we've built a complete Deep Research Agent that demonstrates:

1. **Structured Outputs**: Using Pydantic models to enforce response schemas
2. **Research Planning**: Converting queries into specific searches
3. **Parallel Execution**: Using asyncio for efficiency
4. **Report Synthesis**: Combining results into comprehensive reports
5. **Email Delivery**: Sending formatted reports
6. **Enhancements**: Clarifying questions and report evaluation

### Next Steps:

1. **Deploy with Gradio**: Create a user-friendly web interface (see README.md)
2. **Add Real Web Search**: Use OpenAI's hosted WebSearchTool for actual searches
3. **Implement Guardrails**: Add input validation to protect the system
4. **Make it Autonomous**: Let the agent decide when to do more searches
5. **Custom Data Sources**: Add proprietary data beyond web search

### Cost Considerations:

- This demo uses simulated searches (free)
- Real OpenAI Web Search Tool: ~$0.025 per search
- 3 searches = ~$0.08, 20 searches = ~$0.50
- Monitor usage and set appropriate limits!

**Congratulations!** You've built a production-ready Deep Research Agent! 🎉
